# 04 SOKE Gemma 3 270M LoRA LM

Colab runner for a single causal Gemma LM over SOKE-style flattened motion tokens. This notebook expects the SOKE motion-code JSONL files to already exist in the Drive bundle, then copies them to `/content` for training and syncs adapters/logs back to Drive.

In [ ]:
# Run configuration: edit this cell first in Colab.
from pathlib import Path
import os
import shlex
import subprocess
import sys

DRIVE_BUNDLE_ROOT = Path(os.environ.get(
    'SOKE_GEMMA_DRIVE_BUNDLE_ROOT',
    '/content/drive/MyDrive/folder/COLAB/Tokenizer/04_soke_gemma3_270m_lora_lm',
))
LOCAL_BUNDLE_ROOT = Path(os.environ.get('SOKE_GEMMA_LOCAL_BUNDLE_ROOT', '/content/04_soke_gemma3_270m_lora_lm'))
RUN_NAME = os.environ.get('SOKE_GEMMA_RUN_NAME', 'gemma3_270m_lora_soke_flat_lm')
BASE_MODEL = os.environ.get('SOKE_GEMMA_BASE_MODEL', 'google/gemma-3-270m')
RUN_STAGES = [s.strip() for s in os.environ.get('SOKE_GEMMA_RUN_STAGES', 'lm_instruct').split(',') if s.strip()]
AUTO_INIT_INSTRUCT_FROM_PRETRAIN = int(os.environ.get('SOKE_GEMMA_AUTO_INIT_INSTRUCT_FROM_PRETRAIN', '1'))
INSTRUCT_INIT_ADAPTER_NAME = os.environ.get('SOKE_GEMMA_INSTRUCT_INIT_ADAPTER_NAME', 'last_adapter')  # last_adapter or best_adapter
RESUME_TRAINING = int(os.environ.get('SOKE_GEMMA_RESUME_TRAINING', '1'))
RESUME_ADAPTER_NAME = os.environ.get('SOKE_GEMMA_RESUME_ADAPTER_NAME', 'last_adapter')

# Batch strategy: edit HARDWARE_BATCH_SIZE per GPU memory; virtual batch is HARDWARE_BATCH_SIZE * GRAD_ACCUM.
HARDWARE_BATCH_SIZE = int(os.environ.get('SOKE_GEMMA_HARDWARE_BATCH_SIZE', os.environ.get('SOKE_GEMMA_BATCH_SIZE', '64')))
GRAD_ACCUM = int(os.environ.get('SOKE_GEMMA_GRAD_ACCUM', '2'))
VIRTUAL_BATCH_SIZE = HARDWARE_BATCH_SIZE * GRAD_ACCUM
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# SOKE-style stage schedule: 150 epochs pretrain, then 150 epochs instruct.
EPOCHS_BY_STAGE = {
    'lm_pretrain': int(os.environ.get('SOKE_GEMMA_PRETRAIN_EPOCHS', os.environ.get('SOKE_GEMMA_EPOCHS', '150'))),
    'lm_instruct': int(os.environ.get('SOKE_GEMMA_INSTRUCT_EPOCHS', os.environ.get('SOKE_GEMMA_EPOCHS', '150'))),
}
if not RUN_STAGES:
    raise ValueError('SOKE_GEMMA_RUN_STAGES must contain at least one stage.')
bad_stages = [stage for stage in RUN_STAGES if stage not in EPOCHS_BY_STAGE]
if bad_stages:
    raise ValueError(f'Unknown SOKE_GEMMA_RUN_STAGES entries: {bad_stages}')
NUM_WORKERS = int(os.environ.get('SOKE_GEMMA_NUM_WORKERS', '8'))
PIN_MEMORY = int(os.environ.get('SOKE_GEMMA_PIN_MEMORY', '1'))
PERSISTENT_WORKERS = int(os.environ.get('SOKE_GEMMA_PERSISTENT_WORKERS', '1'))
PREFETCH_FACTOR = int(os.environ.get('SOKE_GEMMA_PREFETCH_FACTOR', '4'))
TF32 = int(os.environ.get('SOKE_GEMMA_TF32', '1'))
TORCH_COMPILE = int(os.environ.get('SOKE_GEMMA_TORCH_COMPILE', '0'))
TORCH_COMPILE_MODE = os.environ.get('SOKE_GEMMA_TORCH_COMPILE_MODE', 'reduce-overhead')
MAX_SEQ_LEN = int(os.environ.get('SOKE_GEMMA_MAX_SEQ_LEN', '1024'))
MAX_TRAIN_LOGICAL_ROWS = int(os.environ.get('SOKE_GEMMA_MAX_TRAIN_LOGICAL_ROWS', '0'))
MAX_VAL_LOGICAL_ROWS = int(os.environ.get('SOKE_GEMMA_MAX_VAL_LOGICAL_ROWS', '2048'))
EVAL_MAX_BATCHES = int(os.environ.get('SOKE_GEMMA_EVAL_MAX_BATCHES', '0'))
GRADIENT_CHECKPOINTING = int(os.environ.get('SOKE_GEMMA_GRADIENT_CHECKPOINTING', '1'))
ATTN_IMPLEMENTATION = os.environ.get('SOKE_GEMMA_ATTN_IMPLEMENTATION', 'auto')  # auto, eager, sdpa, flash_attention_2
SDPA_KERNEL = os.environ.get('SOKE_GEMMA_SDPA_KERNEL', 'auto')  # auto, flash, mem_efficient, math; only used for sdpa
VALIDATION_EVERY_EPOCHS = int(os.environ.get('SOKE_GEMMA_VALIDATION_EVERY_EPOCHS', '5'))
MOTION_VAL_ENABLED = int(os.environ.get('SOKE_GEMMA_MOTION_VAL_ENABLED', '1'))
MOTION_VAL_EVERY_EPOCHS = int(os.environ.get('SOKE_GEMMA_MOTION_VAL_EVERY_EPOCHS', '5'))
MOTION_VAL_FRACTION = float(os.environ.get('SOKE_GEMMA_MOTION_VAL_FRACTION', '0.10'))
MOTION_VAL_MAX_ROWS = int(os.environ.get('SOKE_GEMMA_MOTION_VAL_MAX_ROWS', '256'))
MOTION_VAL_MAX_NEW_TOKENS = int(os.environ.get('SOKE_GEMMA_MOTION_VAL_MAX_NEW_TOKENS', '384'))
INSPECT_N = int(os.environ.get('SOKE_GEMMA_INSPECT_N', '3'))
INSPECT_ROW_LIMIT = int(os.environ.get('SOKE_GEMMA_INSPECT_ROW_LIMIT', '0'))
INSPECT_TOKENIZER_AUDIT = int(os.environ.get('SOKE_GEMMA_INSPECT_TOKENIZER_AUDIT', '0'))

LR = float(os.environ.get('SOKE_GEMMA_LR', '2e-4'))
MOTION_TOKEN_LR = float(os.environ.get('SOKE_GEMMA_MOTION_TOKEN_LR', '2e-5'))
ETA_MIN = float(os.environ.get('SOKE_GEMMA_ETA_MIN', '1e-6'))
DTYPE = os.environ.get('SOKE_GEMMA_DTYPE', 'bf16')  # bf16, fp16, fp32
LORA_R = int(os.environ.get('SOKE_GEMMA_LORA_R', '64'))
LORA_ALPHA = int(os.environ.get('SOKE_GEMMA_LORA_ALPHA', '128'))
LORA_DROPOUT = float(os.environ.get('SOKE_GEMMA_LORA_DROPOUT', '0.05'))
LORA_TARGET_MODULES = os.environ.get('SOKE_GEMMA_LORA_TARGET_MODULES', 'auto')
TRAIN_MOTION_TOKEN_EMBEDDINGS = int(os.environ.get('SOKE_GEMMA_TRAIN_MOTION_TOKEN_EMBEDDINGS', '1'))
RANDOM_DROP = int(os.environ.get('SOKE_GEMMA_RANDOM_DROP', '1'))
SEED = int(os.environ.get('SOKE_GEMMA_SEED', '42'))

SYNC_TO_DRIVE = int(os.environ.get('SOKE_GEMMA_SYNC_TO_DRIVE', '1'))
SAVE_EVERY_EPOCHS = int(os.environ.get('SOKE_GEMMA_SAVE_EVERY_EPOCHS', '1'))
KEEP_LOCAL_EPOCH_SAVES = int(os.environ.get('SOKE_GEMMA_KEEP_LOCAL_EPOCH_SAVES', '5'))
KEEP_DRIVE_EPOCH_SAVES = int(os.environ.get('SOKE_GEMMA_KEEP_DRIVE_EPOCH_SAVES', '5'))
DRIVE_SYNC_RETRIES = int(os.environ.get('SOKE_GEMMA_DRIVE_SYNC_RETRIES', '5'))
DRIVE_SYNC_SLEEP_SEC = float(os.environ.get('SOKE_GEMMA_DRIVE_SYNC_SLEEP_SEC', '20'))
DRIVE_SYNC_DELETE = int(os.environ.get('SOKE_GEMMA_DRIVE_SYNC_DELETE', '1'))
DRIVE_RUN_ROOT_BASE = Path(os.environ.get('SOKE_GEMMA_DRIVE_RUN_ROOT', str(DRIVE_BUNDLE_ROOT / 'outputs' / 'runs' / RUN_NAME)))
LOCAL_RUN_ROOT_BASE = LOCAL_BUNDLE_ROOT / 'outputs' / 'runs' / RUN_NAME
CLOSE_RUNTIME = os.environ.get('SOKE_GEMMA_CLOSE_RUNTIME', '0') == '1'

print({
    'DRIVE_BUNDLE_ROOT': str(DRIVE_BUNDLE_ROOT),
    'LOCAL_BUNDLE_ROOT': str(LOCAL_BUNDLE_ROOT),
    'RUN_NAME': RUN_NAME,
    'BASE_MODEL': BASE_MODEL,
    'RUN_STAGES': RUN_STAGES,
    'INSTRUCT_INIT_ADAPTER_NAME': INSTRUCT_INIT_ADAPTER_NAME,
    'RESUME_TRAINING': RESUME_TRAINING,
    'RESUME_ADAPTER_NAME': RESUME_ADAPTER_NAME,
    'EPOCHS_BY_STAGE': EPOCHS_BY_STAGE,
    'HARDWARE_BATCH_SIZE': HARDWARE_BATCH_SIZE,
    'GRAD_ACCUM': GRAD_ACCUM,
    'VIRTUAL_BATCH_SIZE': VIRTUAL_BATCH_SIZE,
    'MAX_SEQ_LEN': MAX_SEQ_LEN,
    'NUM_WORKERS': NUM_WORKERS,
    'PIN_MEMORY': PIN_MEMORY,
    'PERSISTENT_WORKERS': PERSISTENT_WORKERS,
    'PREFETCH_FACTOR': PREFETCH_FACTOR,
    'TF32': TF32,
    'TORCH_COMPILE': TORCH_COMPILE,
    'TORCH_COMPILE_MODE': TORCH_COMPILE_MODE,
    'GRADIENT_CHECKPOINTING': GRADIENT_CHECKPOINTING,
    'ATTN_IMPLEMENTATION': ATTN_IMPLEMENTATION,
    'SDPA_KERNEL': SDPA_KERNEL,
    'VALIDATION_EVERY_EPOCHS': VALIDATION_EVERY_EPOCHS,
    'MOTION_VAL_ENABLED': MOTION_VAL_ENABLED,
    'MOTION_VAL_EVERY_EPOCHS': MOTION_VAL_EVERY_EPOCHS,
    'MOTION_VAL_FRACTION': MOTION_VAL_FRACTION,
    'MOTION_VAL_MAX_ROWS': MOTION_VAL_MAX_ROWS,
    'INSPECT_N': INSPECT_N,
    'INSPECT_ROW_LIMIT': INSPECT_ROW_LIMIT,
    'INSPECT_TOKENIZER_AUDIT': INSPECT_TOKENIZER_AUDIT,
    'LR': LR,
    'MOTION_TOKEN_LR': MOTION_TOKEN_LR,
    'DTYPE': DTYPE,
    'LORA_R': LORA_R,
    'LORA_ALPHA': LORA_ALPHA,
    'TRAIN_MOTION_TOKEN_EMBEDDINGS': TRAIN_MOTION_TOKEN_EMBEDDINGS,
    'SYNC_TO_DRIVE': SYNC_TO_DRIVE,
    'SAVE_EVERY_EPOCHS': SAVE_EVERY_EPOCHS,
    'KEEP_LOCAL_EPOCH_SAVES': KEEP_LOCAL_EPOCH_SAVES,
    'KEEP_DRIVE_EPOCH_SAVES': KEEP_DRIVE_EPOCH_SAVES,
    'DRIVE_RUN_ROOT_BASE': str(DRIVE_RUN_ROOT_BASE),
})

In [ ]:
# Install runtime dependencies.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers>=4.53.0', 'accelerate>=1.8.0', 'peft>=0.19.0',
    'torchao>=0.16.0', 'safetensors', 'sentencepiece', 'pandas', 'matplotlib'
], check=True)

In [ ]:
# Mount Drive in Colab when available.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', repr(exc))

# Optional Hugging Face auth. Put an HF_TOKEN secret in Colab after accepting the Gemma license.
if not os.environ.get('HF_TOKEN') and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    try:
        from google.colab import userdata
        hf_secret = userdata.get('HF_TOKEN')
        if hf_secret:
            os.environ['HF_TOKEN'] = hf_secret
            print('HF_TOKEN loaded from Colab secrets.')
    except Exception as exc:
        print('HF_TOKEN secret not loaded:', repr(exc))

In [ ]:
# Copy scripts, instructions, and prebuilt JSONL data from Drive to local Colab storage.
LOCAL_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
if not DRIVE_BUNDLE_ROOT.exists():
    raise FileNotFoundError(f'Missing Drive bundle: {DRIVE_BUNDLE_ROOT}')

def run(cmd):
    cmd = [str(x) for x in cmd]
    print(' '.join(shlex.quote(x) for x in cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        tail.append(line.rstrip())
        if len(tail) > 80:
            tail = tail[-80:]
    ret = proc.wait()
    if ret:
        print('\nSubprocess failed. Last output lines:')
        print('\n'.join(tail[-80:]))
        raise subprocess.CalledProcessError(ret, cmd)

for name in ['scripts', 'instructions']:
    src = DRIVE_BUNDLE_ROOT / name
    if not src.exists():
        raise FileNotFoundError(f'Missing Drive bundle directory: {src}')
    run(['rsync', '-a', '--delete', str(src) + '/', str(LOCAL_BUNDLE_ROOT / name) + '/'])

code_src = DRIVE_BUNDLE_ROOT / 'outputs' / 'soke_motion_codes'
code_dst = LOCAL_BUNDLE_ROOT / 'outputs' / 'soke_motion_codes'
if not code_src.exists():
    raise FileNotFoundError(f'Missing prebuilt SOKE code directory on Drive: {code_src}')
code_dst.parent.mkdir(parents=True, exist_ok=True)
run(['rsync', '-a', '--delete', str(code_src) + '/', str(code_dst) + '/'])

required_code_files = ['part_codecs.json', 'train_soke_motion_codes.jsonl', 'val_soke_motion_codes.jsonl']
for filename in required_code_files:
    path = code_dst / filename
    if not path.exists():
        raise FileNotFoundError(f'Missing prebuilt SOKE code file: {path}')
    if path.suffix == '.jsonl':
        print(path, 'lines=', sum(1 for _ in path.open('r', encoding='utf-8')))
    else:
        print(path, 'size=', path.stat().st_size)

In [ ]:
# Inspect random rendered SOKE-style LM examples before training.
cmd = [
    sys.executable, str(LOCAL_BUNDLE_ROOT / 'scripts' / 'inspect_soke_gemma_dataset.py'),
    '--code-root', str(LOCAL_BUNDLE_ROOT / 'outputs' / 'soke_motion_codes'),
    '--instructions-root', str(LOCAL_BUNDLE_ROOT / 'instructions'),
    '--split', 'train', '--stage', RUN_STAGES[0], '--n', str(INSPECT_N), '--seed', str(SEED),
    '--row-limit', str(INSPECT_ROW_LIMIT),
]
if INSPECT_TOKENIZER_AUDIT:
    cmd += ['--tokenizer', BASE_MODEL]
run(cmd)

In [ ]:
# Train Gemma LoRA stages. Resume current stage first; otherwise instruction tuning initializes from the selected pretrain adapter.
def adapter_ready(path):
    path = Path(path)
    has_config = (path / 'adapter_config.json').exists()
    has_weights = (path / 'adapter_model.safetensors').exists() or (path / 'adapter_model.bin').exists()
    return has_config and has_weights

def stage_resume_ready(path):
    path = Path(path)
    return adapter_ready(path / RESUME_ADAPTER_NAME) and (path / 'last_train_state.pt').exists()

def status_epoch(path):
    path = Path(path)
    status_path = path / 'latest_status.json'
    if not status_path.exists():
        return 0
    try:
        import json
        payload = json.loads(status_path.read_text())
        return int(payload.get('epoch') or payload.get('resume_epoch') or 0)
    except Exception:
        return 0

def maybe_sync_resume_from_drive(local_run_root, drive_run_root):
    if not RESUME_TRAINING:
        return
    if not stage_resume_ready(drive_run_root):
        return
    local_epoch = status_epoch(local_run_root)
    drive_epoch = status_epoch(drive_run_root)
    if (not stage_resume_ready(local_run_root)) or drive_epoch > local_epoch:
        local_run_root.parent.mkdir(parents=True, exist_ok=True)
        print(f'Syncing resumable stage run from Drive: {drive_run_root} -> {local_run_root} (drive_epoch={drive_epoch}, local_epoch={local_epoch})')
        run(['rsync', '-a', '--delete', str(drive_run_root) + '/', str(local_run_root) + '/'])
    else:
        print(f'Keeping local resumable stage run: {local_run_root} (local_epoch={local_epoch}, drive_epoch={drive_epoch})')

previous_pretrain_adapter = None
for stage in RUN_STAGES:
    if stage not in EPOCHS_BY_STAGE:
        raise ValueError(f'Unknown stage: {stage}')
    stage_epochs = EPOCHS_BY_STAGE[stage]
    local_run_root = LOCAL_RUN_ROOT_BASE / stage
    drive_run_root = DRIVE_RUN_ROOT_BASE / stage
    maybe_sync_resume_from_drive(local_run_root, drive_run_root)

    resume_current_stage = bool(RESUME_TRAINING and stage_resume_ready(local_run_root))
    init_adapter = None
    if resume_current_stage:
        print('Resuming current stage from:', local_run_root / RESUME_ADAPTER_NAME)
        print('Resume state:', local_run_root / 'last_train_state.pt')
    elif stage == 'lm_instruct' and AUTO_INIT_INSTRUCT_FROM_PRETRAIN:
        candidate = previous_pretrain_adapter or (LOCAL_RUN_ROOT_BASE / 'lm_pretrain' / INSTRUCT_INIT_ADAPTER_NAME)
        if not adapter_ready(candidate):
            drive_candidate = DRIVE_RUN_ROOT_BASE / 'lm_pretrain' / INSTRUCT_INIT_ADAPTER_NAME
            if adapter_ready(drive_candidate):
                candidate.parent.mkdir(parents=True, exist_ok=True)
                run(['rsync', '-a', str(drive_candidate) + '/', str(candidate) + '/'])
        if adapter_ready(candidate):
            init_adapter = candidate
            print('Initializing lm_instruct from:', init_adapter)
        else:
            print(f'No valid pretrain {INSTRUCT_INIT_ADAPTER_NAME} found locally or on Drive; lm_instruct will start from base Gemma.')

    cmd = [
        sys.executable, str(LOCAL_BUNDLE_ROOT / 'scripts' / 'train_gemma_soke_lora.py'),
        '--code-root', str(LOCAL_BUNDLE_ROOT / 'outputs' / 'soke_motion_codes'),
        '--instructions-root', str(LOCAL_BUNDLE_ROOT / 'instructions'),
        '--run-root', str(local_run_root),
        '--base-model', BASE_MODEL,
        '--stage', stage,
        '--epochs', str(stage_epochs),
        '--resume-training', str(RESUME_TRAINING),
        '--resume-adapter-name', RESUME_ADAPTER_NAME,
        '--batch-size', str(HARDWARE_BATCH_SIZE),
        '--grad-accum', str(GRAD_ACCUM),
        '--num-workers', str(NUM_WORKERS),
        '--pin-memory', str(PIN_MEMORY),
        '--persistent-workers', str(PERSISTENT_WORKERS),
        '--prefetch-factor', str(PREFETCH_FACTOR),
        '--tf32', str(TF32),
        '--torch-compile', str(TORCH_COMPILE),
        '--torch-compile-mode', TORCH_COMPILE_MODE,
        '--max-seq-len', str(MAX_SEQ_LEN),
        '--gradient-checkpointing', str(GRADIENT_CHECKPOINTING),
        '--attn-implementation', ATTN_IMPLEMENTATION,
        '--sdpa-kernel', SDPA_KERNEL,
        '--max-train-logical-rows', str(MAX_TRAIN_LOGICAL_ROWS),
        '--max-val-logical-rows', str(MAX_VAL_LOGICAL_ROWS),
        '--eval-max-batches', str(EVAL_MAX_BATCHES),
        '--validation-every-epochs', str(VALIDATION_EVERY_EPOCHS),
        '--motion-val-enabled', str(MOTION_VAL_ENABLED),
        '--motion-val-every-epochs', str(MOTION_VAL_EVERY_EPOCHS),
        '--motion-val-fraction', str(MOTION_VAL_FRACTION),
        '--motion-val-max-rows', str(MOTION_VAL_MAX_ROWS),
        '--motion-val-max-new-tokens', str(MOTION_VAL_MAX_NEW_TOKENS),
        '--lr', str(LR),
        '--motion-token-lr', str(MOTION_TOKEN_LR),
        '--eta-min', str(ETA_MIN),
        '--dtype', DTYPE,
        '--lora-r', str(LORA_R),
        '--lora-alpha', str(LORA_ALPHA),
        '--lora-dropout', str(LORA_DROPOUT),
        '--lora-target-modules', LORA_TARGET_MODULES,
        '--train-motion-token-embeddings', str(TRAIN_MOTION_TOKEN_EMBEDDINGS),
        '--random-drop', str(RANDOM_DROP),
        '--seed', str(SEED),
        '--save-every-epochs', str(SAVE_EVERY_EPOCHS),
        '--keep-local-epoch-saves', str(KEEP_LOCAL_EPOCH_SAVES),
        '--sync-to-drive', str(SYNC_TO_DRIVE),
        '--drive-sync-dest', str(drive_run_root),
        '--drive-sync-retries', str(DRIVE_SYNC_RETRIES),
        '--drive-sync-sleep-sec', str(DRIVE_SYNC_SLEEP_SEC),
        '--drive-sync-delete', str(DRIVE_SYNC_DELETE),
        '--keep-drive-epoch-saves', str(KEEP_DRIVE_EPOCH_SAVES),
    ]
    if init_adapter is not None and not resume_current_stage:
        cmd += ['--init-adapter', str(init_adapter)]
    run(cmd)
    if stage == 'lm_pretrain':
        previous_pretrain_adapter = local_run_root / INSTRUCT_INIT_ADAPTER_NAME


In [ ]:
# Show saved history.
import pandas as pd
for stage in RUN_STAGES:
    local_run_root = LOCAL_RUN_ROOT_BASE / stage
    drive_run_root = DRIVE_RUN_ROOT_BASE / stage
    history_path = local_run_root / 'history.csv'
    if not history_path.exists():
        drive_history_path = drive_run_root / 'history.csv'
        if drive_history_path.exists():
            history_path = drive_history_path
    print('\nStage:', stage)
    if history_path.exists():
        display(pd.read_csv(history_path))
    else:
        print('No history.csv found yet.')
    print('Local run root:', local_run_root)
    print('Drive run root:', drive_run_root)

In [ ]:
# Optional Colab runtime close. Set SOKE_GEMMA_CLOSE_RUNTIME=1 when launching unattended.
if CLOSE_RUNTIME:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print('Runtime close skipped:', repr(exc))
else:
    print('Runtime left active. Set SOKE_GEMMA_CLOSE_RUNTIME=1 to close automatically.')